# Multi-hop Proxy Chaining

Route traffic through multiple sequential proxy nodes using go-gost chains, proxychains-ng, or SSH ProxyJump.

## Build a go-gost chain config

In [ ]:
import yaml

def gost_chain_config(listen_port, hops):
    """
    hops: list of {"addr": "host:port", "type": "socks5"|"http", "user": ..., "pass": ...}
    """
    chain_hops = []
    for i, hop in enumerate(hops):
        node = {"name": f"proxy{i+1}", "addr": hop["addr"],
                "connector": {"type": hop.get("type", "socks5")}}
        if "user" in hop:
            node["connector"]["auth"] = {"username": hop["user"], "password": hop["pass"]}
        chain_hops.append({"name": f"hop{i+1}", "nodes": [node]})

    return {
        "log": {"level": "info"},
        "services": [{
            "name": "entry",
            "addr": f":{listen_port}",
            "handler": {"type": "http+socks5"},
            "listener": {"type": "tcp"},
            "chain": "multi-hop",
        }],
        "chains": [{"name": "multi-hop", "hops": chain_hops}],
    }

cfg = gost_chain_config(8080, [
    {"addr": "proxy1.example.com:1080", "type": "socks5", "user": "u1", "pass": "p1"},
    {"addr": "proxy2.example.com:8080", "type": "http",   "user": "u2", "pass": "p2"},
    {"addr": "proxy3.example.com:1080", "type": "socks5"},
])
print(yaml.dump(cfg, default_flow_style=False))

## Generate proxychains.conf

In [ ]:
def proxychains_conf(proxies, chain_type="strict_chain", proxy_dns=True):
    lines = [f"# proxychains-ng configuration — {chain_type}",
             chain_type, "quiet_mode"]
    if proxy_dns:
        lines.append("proxy_dns")
    lines += ["", "[ProxyList]"]
    for p in proxies:
        line = f"{p['type']:<8} {p['host']:<30} {p['port']}"
        if "user" in p:
            line += f"  {p['user']} {p['pass']}"
        lines.append(line)
    return "\n".join(lines)

print(proxychains_conf([
    {"type": "socks5", "host": "proxy1.example.com", "port": 1080, "user": "u1", "pass": "p1"},
    {"type": "socks5", "host": "proxy2.example.com", "port": 1080},
    {"type": "http",   "host": "proxy3.example.com", "port": 8080},
]))

## SSH ProxyJump commands

In [ ]:
def ssh_proxyjump(jumps, final_host, user="deploy"):
    jump_str = ",".join(f"{j['user']}@{j['host']}" for j in jumps)
    return f"ssh -J {jump_str} {user}@{final_host}"

def ssh_config_entry(host, jumps, user="deploy"):
    lines = [f"Host {host}", f"    User {user}",
             f"    ProxyJump {','.join(j['host'] for j in jumps)}"]
    return "\n".join(lines)

jumps = [{"user": "ops", "host": "bastion1.example.com"},
         {"user": "ops", "host": "pivot.internal"}]

print("# One-liner:")
print(ssh_proxyjump(jumps, "target.internal"))
print()
print("# ~/.ssh/config entry:")
print(ssh_config_entry("target.internal", jumps))

## Verify chain exit IP

In [ ]:
chain_test_cmds = [
    "# Via go-gost (running on :8080)",
    "curl -s --proxy socks5h://localhost:8080 https://api.ipify.org",
    "",
    "# Via proxychains",
    "proxychains curl -s https://api.ipify.org",
    "",
    "# Via SSH tunnel (port 8080 bound locally via SSH -D)",
    "ssh -D 8080 -N user@bastion.example.com &",
    "curl -s --proxy socks5h://localhost:8080 https://api.ipify.org",
]
for line in chain_test_cmds:
    print(line)